# Data Processing & Analysis - Complete Pipeline

Notebook unifié pour le traitement complet des données du corpus de messages sur l'accueil de réfugiés.

**Pipeline complet** :
1. Chargement & Exploration
2. Contrôle Qualité
3. Nettoyage
4. Feature Engineering
5. Analyse Statistique
6. Analyse Emotionnelle Basique
7. Sauvegarde

**Input** : 3 feuilles Excel (~9,000 messages)

**Output** :
- `data/processed/messages_unified.csv` - Dataset complet unifié
- `data/processed/sample_messages_100.csv` - Échantillon pour tests
- `data/processed/dataset_statistics.json` - Statistiques complètes

In [1]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter
from datetime import datetime
import json
import re
import warnings
warnings.filterwarnings('ignore')

# Configuration affichage
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)

print('✓ Bibliothèques chargées')

✓ Bibliothèques chargées


## 1. CHARGEMENT & EXPLORATION

Chargement des 3 feuilles Excel et vue d'ensemble des données brutes.

In [2]:
# Trouver le fichier Excel
data_dir = Path('../data/raw')
excel_files = list(data_dir.glob('*.xlsx'))

if not excel_files:
    raise FileNotFoundError('Aucun fichier Excel trouvé dans data/raw/')

data_path = excel_files[0]
print(f'📂 Chargement : {data_path.name}\n')

# Charger les 3 feuilles
f1 = pd.read_excel(data_path, sheet_name='Familles 1')
f2 = pd.read_excel(data_path, sheet_name='Familles 2')
np_df = pd.read_excel(data_path, sheet_name='Nouvelle Plateforme')

print(f'✓ Familles 1: {len(f1):,} messages, {len(f1.columns)} colonnes')
print(f'✓ Familles 2: {len(f2):,} messages, {len(f2.columns)} colonnes')
print(f'✓ Nouvelle Plateforme: {len(np_df):,} messages, {len(np_df.columns)} colonnes')
print(f'\n📊 Total : {len(f1) + len(f2) + len(np_df):,} messages')

📂 Chargement : Te?le?com Paristech Re?cup ération des données.xlsx

✓ Familles 1: 1,782 messages, 8 colonnes
✓ Familles 2: 6,810 messages, 11 colonnes
✓ Nouvelle Plateforme: 477 messages, 12 colonnes

📊 Total : 9,069 messages


In [3]:
# Aperçu des colonnes
print('Colonnes Familles 1 :', list(f1.columns))
print('\nColonnes Familles 2 :', list(f2.columns))
print('\nColonnes Nouvelle Plateforme :', list(np_df.columns))

Colonnes Familles 1 : ["Trois types d'études", 'CP', 'Motivations / Pourquoi CALM', "Type d'espace mis à disposition ", 'Parlez nous de vous', "Temps d'accueil", 'avec enfants ? ', 'Partage de réseau/métier ? ']

Colonnes Familles 2 : ['CP', 'Age', 'Motivations / Pourquoi CALM', "Type d'espace mis à disposition ", 'Taille', 'Parlez nous de vous', "Temps d'accueil", 'avec enfants ? ', 'Enseignement métier / partage réseau', "Capacité d'accueil", 'Autres infos']

Colonnes Nouvelle Plateforme : ['CP', 'Date de naissance', 'Motivations / Pourquoi CALM', "Type d'espace mis à disposition ", 'Taille', 'Parlez nous de vous', "Temps d'accueil", 'avec enfants ? ', 'Enseignement métier', 'Partage de réseau', "Capacité d'accueil", 'Autres infos']


In [4]:
# Aperçu des données
print('=== Exemple de messages (Familles 2) ===\n')
display(f2[['CP', 'Age', 'Motivations / Pourquoi CALM', "Temps d'accueil"]].head(3))

=== Exemple de messages (Familles 2) ===



,CP,Age,Motivations / Pourquoi CALM,Temps d'accueil
0,27370,32,je recherche depuis plusieurs semaines une association aidant les réfugiés provenant de pays en ...,4-6 mois
1,37100,60,je me sens directement concernée par les évennements et je souhaite m'impliquer en tant que pers...,Autant que nécessaire
2,62200,44 ans,"Construire un lien d'amitié, Enseigner le français, Aider administrativement (étude, job...). J'...",Autant que nécessaire


## 2. CONTRÔLE QUALITÉ

Analyse de la qualité des données brutes.

In [5]:
# Valeurs manquantes
print('=== VALEURS MANQUANTES ===\n')
print('Familles 1 :')
print(f1.isnull().sum()[f1.isnull().sum() > 0])
print('\nFamilles 2 :')
print(f2.isnull().sum()[f2.isnull().sum() > 0])
print('\nNouvelle Plateforme :')
print(np_df.isnull().sum()[np_df.isnull().sum() > 0])

=== VALEURS MANQUANTES ===

Familles 1 :
Trois types d'études                1779
CP                                    47
Motivations / Pourquoi CALM          301
Type d'espace mis à disposition       54
Parlez nous de vous                  214
Temps d'accueil                      115
avec enfants ?                        80
Partage de réseau/métier ?           343
dtype: int64

Familles 2 :
CP                                         4
Age                                        3
Motivations / Pourquoi CALM                7
Type d'espace mis à disposition            1
Taille                                     4
Parlez nous de vous                        5
Temps d'accueil                           57
avec enfants ?                             3
Enseignement métier / partage réseau       4
Capacité d'accueil                      2207
Autres infos                            3678
dtype: int64

Nouvelle Plateforme :
CP                              23
Motivations / Pourquoi CALM      1
Tai

In [6]:
# Statistiques sur le champ principal (Motivations)
motiv_f1 = f1['Motivations / Pourquoi CALM'].dropna()
motiv_f2 = f2['Motivations / Pourquoi CALM'].dropna()
motiv_np = np_df['Motivations / Pourquoi CALM'].dropna()

print('=== CHAMP "MOTIVATIONS" ===\n')
print(f'Familles 1 : {len(motiv_f1):,}/{len(f1):,} ({len(motiv_f1)/len(f1)*100:.1f}%) non vides')
print(f'Familles 2 : {len(motiv_f2):,}/{len(f2):,} ({len(motiv_f2)/len(f2)*100:.1f}%) non vides')
print(f'Nouvelle Plateforme : {len(motiv_np):,}/{len(np_df):,} ({len(motiv_np)/len(np_df)*100:.1f}%) non vides')

all_motivations = pd.concat([motiv_f1, motiv_f2, motiv_np])
print(f'\n📝 Total motivations non vides : {len(all_motivations):,}')
print(f'Longueur moyenne : {all_motivations.str.len().mean():.0f} caractères')
print(f'Longueur médiane : {all_motivations.str.len().median():.0f} caractères')

=== CHAMP "MOTIVATIONS" ===

Familles 1 : 1,481/1,782 (83.1%) non vides
Familles 2 : 6,803/6,810 (99.9%) non vides
Nouvelle Plateforme : 476/477 (99.8%) non vides

📝 Total motivations non vides : 8,760
Longueur moyenne : 174 caractères
Longueur médiane : 118 caractères


## 3. NETTOYAGE

Standardisation des colonnes, nettoyage du texte, et unification des 3 datasets.

In [7]:
# Mapping des colonnes pour standardisation
column_mapping = {
    'familles_1': {
        'CP': 'postal_code',
        'Motivations / Pourquoi CALM': 'motivation_text',
        "Type d'espace mis à disposition ": 'space_type',
        'Parlez nous de vous': 'personal_description',
        "Temps d'accueil": 'hosting_duration',
        'avec enfants ? ': 'accepts_children',
        'Partage de réseau/métier ? ': 'shares_network'
    },
    'familles_2': {
        'CP': 'postal_code',
        'Age': 'age',
        'Motivations / Pourquoi CALM': 'motivation_text',
        "Type d'espace mis à disposition ": 'space_type',
        'Taille': 'space_size_m2',
        'Parlez nous de vous': 'personal_description',
        "Temps d'accueil": 'hosting_duration',
        'avec enfants ? ': 'accepts_children',
        'Enseignement métier / partage réseau': 'teaches_or_shares',
        "Capacité d'accueil": 'hosting_capacity',
        'Autres infos': 'other_info'
    },
    'nouvelle_plateforme': {
        'CP': 'postal_code',
        'Date de naissance': 'birth_date',
        'Motivations / Pourquoi CALM': 'motivation_text',
        "Type d'espace mis à disposition ": 'space_type',
        'Taille': 'space_size_m2',
        'Parlez nous de vous': 'personal_description',
        "Temps d'accueil": 'hosting_duration',
        'avec enfants ? ': 'accepts_children',
        'Enseignement métier': 'teaches_job',
        'Partage de réseau': 'shares_network',
        "Capacité d'accueil": 'hosting_capacity',
        'Autres infos': 'other_info'
    }
}

# Renommer colonnes
f1_clean = f1.rename(columns=column_mapping['familles_1'])
f2_clean = f2.rename(columns=column_mapping['familles_2'])
np_clean = np_df.rename(columns=column_mapping['nouvelle_plateforme'])

# Ajouter identifiant source
f1_clean['source_sheet'] = 'Familles 1'
f2_clean['source_sheet'] = 'Familles 2'
np_clean['source_sheet'] = 'Nouvelle Plateforme'

print('✓ Colonnes standardisées')

✓ Colonnes standardisées


In [8]:
# Fonction de nettoyage de texte
def clean_text(text):
    if pd.isna(text) or not isinstance(text, str):
        return None
    text = re.sub(r'\s+', ' ', text).strip()
    return text if len(text) > 0 else None

# Nettoyer les champs texte
text_columns = ['motivation_text', 'personal_description', 'other_info', 'space_type']

for df in [f1_clean, f2_clean, np_clean]:
    for col in text_columns:
        if col in df.columns:
            df[col] = df[col].apply(clean_text)

print('✓ Textes nettoyés')

✓ Textes nettoyés


In [9]:
# Unifier les 3 datasets
df = pd.concat([f1_clean, f2_clean, np_clean], ignore_index=True, sort=False)

# Créer IDs uniques
def create_message_id(row, idx):
    prefix = {'Familles 1': 'F1', 'Familles 2': 'F2', 'Nouvelle Plateforme': 'NP'}[row['source_sheet']]
    return f'{prefix}_{idx:04d}'

df['message_id'] = [create_message_id(row, idx) for idx, row in df.iterrows()]

# Réorganiser colonnes
cols = ['message_id', 'source_sheet'] + [c for c in df.columns if c not in ['message_id', 'source_sheet']]
df = df[cols]

print(f'✓ Dataset unifié : {len(df):,} messages, {len(df.columns)} colonnes')

✓ Dataset unifié : 9,069 messages, 17 colonnes


## 4. FEATURE ENGINEERING

Création de nouvelles features et catégorisation.

In [10]:
# Extraire département du code postal (gérant codes non-français)def extract_department(cp):    if pd.isna(cp):        return None        cp_str = str(cp).strip()        # Si commence par une lettre, pas un code postal français    if cp_str and cp_str[0].isalpha():        return None        # Essayer de convertir en int pour nettoyer    try:        cp_int = int(float(cp_str))        cp_str = str(cp_int)    except (ValueError, TypeError):        pass        # Extraire les 2 premiers chiffres    digits = re.findall(r'\d+', cp_str)    if digits and len(digits[0]) >= 2:        return digits[0][:2]        return Nonedef format_postal_code(cp):    '''Formate code postal en gérant codes non-français'''    if pd.isna(cp):        return None        cp_str = str(cp).strip()        # Si contient des lettres, garder tel quel (code non-français)    if any(c.isalpha() for c in cp_str):        return cp_str        # Sinon formater comme code postal français (5 chiffres)    try:        cp_int = int(float(cp_str))        return str(cp_int).zfill(5)    except (ValueError, TypeError):        return cp_strdf['department'] = df['postal_code'].apply(extract_department)df['postal_code'] = df['postal_code'].apply(format_postal_code)french_codes = df[df['department'].notna()].shape[0]total_codes = df[df['postal_code'].notna()].shape[0]print(f'✓ Département extrait : {df["department"].nunique()} départements français uniques')print(f'  Codes postaux français : {french_codes:,}/{total_codes:,}')

ValueError: invalid literal for int() with base 10: 'B4880'

In [ ]:
# Features de texte
df['text_length'] = df['motivation_text'].apply(lambda x: len(x) if pd.notna(x) else 0)
df['has_motivation'] = df['motivation_text'].notna()
df['has_description'] = df['personal_description'].notna()
df['has_other_info'] = df['other_info'].notna()

print(f'✓ Features texte créées')
print(f'  Messages avec motivation : {df["has_motivation"].sum():,} ({df["has_motivation"].mean()*100:.1f}%)')
print(f'  Longueur moyenne : {df["text_length"].mean():.0f} caractères')

In [ ]:
# Catégoriser durée d'accueil
def categorize_duration(duration):
    if pd.isna(duration):
        return 'unknown'
    d = str(duration).lower()
    if 'semaine' in d and '2' in d:
        return 'short'
    elif '3 mois' in d or '3-6' in d or '1-3' in d:
        return 'medium'
    elif '6' in d or '9' in d or '4-6' in d:
        return 'medium-long'
    elif 'an' in d or 'année' in d:
        return 'long'
    elif 'autant' in d or 'limite' in d or 'nécessaire' in d or 'faudra' in d:
        return 'unlimited'
    elif 'urgence' in d:
        return 'emergency'
    else:
        return 'other'

df['duration_category'] = df['hosting_duration'].apply(categorize_duration)

print('✓ Durée catégorisée')
print(df['duration_category'].value_counts())

In [ ]:
# Catégoriser type d'espace
def categorize_space(space):
    if pd.isna(space):
        return 'unknown'
    s = str(space).lower()
    if 'chambre' in s:
        return 'room'
    elif 'studio' in s:
        return 'studio'
    elif 'logement' in s or 'appartement' in s:
        return 'autonomous'
    elif 'canapé' in s or 'salon' in s:
        return 'sofa'
    else:
        return 'other'

df['space_category'] = df['space_type'].apply(categorize_space)

print('✓ Espace catégorisé')
print(df['space_category'].value_counts())

In [ ]:
# Score de qualité (0-100)
def calculate_quality_score(row):
    score = 0
    if row['has_motivation']:
        score += 40
        if row['text_length'] > 100:
            score += 5
        if row['text_length'] > 200:
            score += 5
    if row['has_description']:
        score += 20
    if pd.notna(row['postal_code']):
        score += 10
    if pd.notna(row.get('age')) or pd.notna(row.get('birth_date')):
        score += 5
    if pd.notna(row.get('space_type')):
        score += 5
    if pd.notna(row['hosting_duration']):
        score += 5
    if pd.notna(row.get('accepts_children')):
        score += 5
    return min(score, 100)

df['quality_score'] = df.apply(calculate_quality_score, axis=1)

print('✓ Scores de qualité calculés')
print(f'\nDistribution des scores :')
print(df['quality_score'].describe())
print(f'\nHaute qualité (≥70) : {(df["quality_score"] >= 70).sum():,} messages')

## 5. ANALYSE STATISTIQUE

Statistiques descriptives et visualisations.

In [ ]:
# Distribution géographique
print('=== TOP 10 DÉPARTEMENTS ===\n')
print(df['department'].value_counts().head(10))

plt.figure(figsize=(12, 6))
df['department'].value_counts().head(15).plot(kind='bar')
plt.xlabel('Département')
plt.ylabel('Nombre de messages')
plt.title('Top 15 Départements')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Distribution longueur texte
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
df[df['text_length'] > 0]['text_length'].hist(bins=50, edgecolor='black')
plt.xlabel('Longueur (caractères)')
plt.ylabel('Fréquence')
plt.title('Distribution de la longueur des motivations')

plt.subplot(1, 2, 2)
df[df['text_length'] > 0]['text_length'].plot(kind='box')
plt.ylabel('Longueur (caractères)')
plt.title('Box plot')

plt.tight_layout()
plt.show()

In [ ]:
# Durée et espace
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df['duration_category'].value_counts().plot(kind='barh', ax=axes[0])
axes[0].set_title('Distribution des durées d\'accueil')
axes[0].set_xlabel('Nombre de messages')

df['space_category'].value_counts().plot(kind='barh', ax=axes[1])
axes[1].set_title('Distribution des types d\'espace')
axes[1].set_xlabel('Nombre de messages')

plt.tight_layout()
plt.show()

In [ ]:
# Acceptance enfants
children_counts = df['accepts_children'].dropna().value_counts()

plt.figure(figsize=(8, 6))
children_counts.plot(kind='pie', autopct='%1.1f%%')
plt.title('Acceptance d\'enfants')
plt.ylabel('')
plt.show()

## 6. ANALYSE ÉMOTIONNELLE BASIQUE

Analyse des mots-clés émotionnels et patterns linguistiques.

In [ ]:
# Top mots fréquents
all_text = ' '.join(df[df['has_motivation']]['motivation_text'].astype(str).tolist())
words = re.findall(r'\b[a-zàâäéèêëïîôùûüÿœç]+\b', all_text.lower())
word_freq = Counter(words)

# Stop words français
stop_words = {'le', 'la', 'les', 'un', 'une', 'des', 'de', 'du', 'et', 'est', 'à', 'en',
              'que', 'qui', 'nous', 'vous', 'je', 'il', 'elle', 'dans', 'pour', 'sur',
              'avec', 'par', 'ce', 'se', 'son', 'sa', 'ses', 'au', 'aux', 'mais', 'ou',
              'car', 'ne', 'pas', 'd', 'l', 'j', 'ai', 'avons', 'ont', 'être', 'avoir',
              'mon', 'ma', 'mes', 'notre', 'nos', 'votre', 'vos', 'leur', 'leurs', 'cette',
              'suis', 'sommes', 'peut', 'tout', 'comme', 'bien', 'plus', 'très', 'aussi'}

filtered_words = {w: c for w, c in word_freq.items() if w not in stop_words and len(w) > 2}
top_20 = dict(sorted(filtered_words.items(), key=lambda x: x[1], reverse=True)[:20])

print('=== TOP 20 MOTS ===\n')
for word, count in top_20.items():
    print(f'{word:20s}: {count:5d}')

plt.figure(figsize=(12, 6))
plt.barh(list(top_20.keys()), list(top_20.values()))
plt.xlabel('Fréquence')
plt.title('Top 20 mots les plus fréquents')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Mots-clés émotionnels
emotion_keywords = {
    'aide': 0, 'aider': 0, 'solidarité': 0, 'partage': 0, 'partager': 0,
    'accueil': 0, 'accueillir': 0, 'famille': 0, 'humanité': 0,
    'échange': 0, 'rencontre': 0, 'culture': 0, 'besoin': 0,
    'espoir': 0, 'générosité': 0, 'choqué': 0, 'choquée': 0, 'triste': 0
}

motivations = df[df['has_motivation']]['motivation_text']
for keyword in emotion_keywords:
    emotion_keywords[keyword] = motivations.str.lower().str.contains(keyword, na=False).sum()

sorted_emotions = dict(sorted(emotion_keywords.items(), key=lambda x: x[1], reverse=True))

print('=== MOTS-CLÉS ÉMOTIONNELS ===\n')
for word, count in sorted_emotions.items():
    pct = count/len(motivations)*100
    print(f'{word:15s}: {count:5d} ({pct:4.1f}%)')

plt.figure(figsize=(12, 6))
plt.bar(sorted_emotions.keys(), sorted_emotions.values())
plt.xlabel('Mot-clé')
plt.ylabel('Fréquence')
plt.title('Mots-clés émotionnels dans les motivations')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Exemples de messages par catégorie émotionnelle
print('=== EXEMPLES DE MESSAGES ===\n')

# Message court
short = df[(df['text_length'] > 20) & (df['text_length'] < 100)]['motivation_text'].iloc[0]
print('Court (<100 char) :')
print(f'"{short}"\n')

# Message moyen
medium = df[(df['text_length'] >= 100) & (df['text_length'] < 300)]['motivation_text'].iloc[0]
print('Moyen (100-300 char) :')
print(f'"{medium}"\n')

# Message long
long_msg = df[df['text_length'] >= 300]['motivation_text'].iloc[0]
print('Long (>300 char) :')
print(f'"{long_msg[:300]}..."')

## 7. SAUVEGARDE

Génération des fichiers de sortie.

In [ ]:
# Statistiques complètes
stats = {
    'dataset_info': {
        'total_messages': len(df),
        'source_sheets': df['source_sheet'].value_counts().to_dict(),
        'columns': list(df.columns),
        'generated_date': datetime.now().isoformat()
    },
    'text_content': {
        'messages_with_motivation': int(df['has_motivation'].sum()),
        'messages_with_description': int(df['has_description'].sum()),
        'avg_text_length': float(df['text_length'].mean()),
        'median_text_length': float(df['text_length'].median())
    },
    'geography': {
        'unique_departments': int(df['department'].nunique()),
        'top_10_departments': df['department'].value_counts().head(10).to_dict()
    },
    'hosting': {
        'duration_categories': df['duration_category'].value_counts().to_dict(),
        'space_categories': df['space_category'].value_counts().to_dict()
    },
    'quality': {
        'avg_quality_score': float(df['quality_score'].mean()),
        'high_quality_count': int((df['quality_score'] >= 70).sum())
    },
    'emotion_keywords': sorted_emotions
}

print('✓ Statistiques générées')

In [ ]:
# Sauvegarder les outputs
output_dir = Path('../data/processed')
output_dir.mkdir(exist_ok=True)

# 1. Dataset complet (CSV)
csv_path = output_dir / 'messages_unified.csv'
df.to_csv(csv_path, index=False, encoding='utf-8')
print(f'✓ Dataset complet : {csv_path} ({csv_path.stat().st_size / 1024:.1f} KB)')

# 2. Dataset complet (Parquet - compressé)
parquet_path = output_dir / 'messages_unified.parquet'
df.to_parquet(parquet_path, index=False)
print(f'✓ Dataset Parquet : {parquet_path} ({parquet_path.stat().st_size / 1024:.1f} KB)')

# 3. Échantillon (100 messages haute qualité)
sample_df = df[df['quality_score'] >= 60].sample(n=min(100, len(df)), random_state=42)
sample_path = output_dir / 'sample_messages_100.csv'
sample_df.to_csv(sample_path, index=False, encoding='utf-8')
print(f'✓ Échantillon (100) : {sample_path} ({sample_path.stat().st_size / 1024:.1f} KB)')

# 4. Statistiques JSON
stats_path = output_dir / 'dataset_statistics.json'
with open(stats_path, 'w', encoding='utf-8') as f:
    json.dump(stats, f, indent=2, ensure_ascii=False)
print(f'✓ Statistiques : {stats_path}')

# 5. Mapping colonnes
mapping_path = output_dir / 'column_mapping.json'
with open(mapping_path, 'w', encoding='utf-8') as f:
    json.dump(column_mapping, f, indent=2, ensure_ascii=False)
print(f'✓ Mapping colonnes : {mapping_path}')

In [ ]:
# Résumé final
print('\n' + '='*70)
print('✅ TRAITEMENT COMPLET TERMINÉ')
print('='*70)
print(f'\n📊 Résultats :')
print(f'  • Messages traités : {len(df):,}')
print(f'  • Score qualité moyen : {df["quality_score"].mean():.1f}/100')
print(f'  • Messages haute qualité (≥70) : {(df["quality_score"] >= 70).sum():,}')
print(f'  • Départements couverts : {df["department"].nunique()}')
print(f'\n📁 Fichiers générés :')
print(f'  • {csv_path.name}')
print(f'  • {parquet_path.name}')
print(f'  • {sample_path.name}')
print(f'  • {stats_path.name}')
print(f'\n🎯 Prochaine étape :')
print(f'  → Ouvrir message_viewer.ipynb pour explorer les messages')
print(f'  → Créer emotion_deep_analysis.ipynb pour analyse NLP avancée')
print('='*70)

In [ ]:
# Aperçu final du dataset
print('\nAperçu du dataset final :\n')
display(df[['message_id', 'source_sheet', 'department', 'text_length',
            'duration_category', 'space_category', 'quality_score']].head(10))